# NULLXES CERBER-ULTRA · YOLO26 на Colab A100

Обучение и export. Runtime камеры — на локальной машине (`python -m cerber`).

Порядок: smoke `coco8-seg` → VisDrone detect → Seraphim subset. Отдельные `outputs/<name>/`, без цепочки весов.

In [ ]:
import torch
print("cuda", torch.cuda.is_available())
print("device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "подключи A100 (Runtime → Change runtime type)"

In [ ]:
%pip install -U ultralytics huggingface_hub pyyaml

In [ ]:
import os
from huggingface_hub import login

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if token:
    login(token=token, add_to_git_credential=False)
    print("Hugging Face: logged in")
else:
    print("HF_TOKEN нет — публичные датасеты всё равно качаются")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
# поправь путь, если репозиторий лежит иначе
%cd "/content/drive/MyDrive/NULLXES CERBER ULTRA"
import pathlib, sys
root = pathlib.Path.cwd()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("cwd", root)
assert (root / "cerber" / "__main__.py").is_file(), "это не корень CERBER"

## Smoke: coco8-seg

In [ ]:
!python -m cerber.experiments.train --config configs/experiments/coco8-seg.yaml

## VisDrone detect (`yolo26n.pt`)

Первый запуск скачает VisDrone и сконвертирует аннотации (боксы, не маски).

In [ ]:
!python -m cerber.experiments.train --config configs/experiments/visdrone-n.yaml

In [ ]:
!python -m cerber.experiments.val --config configs/experiments/visdrone-n.yaml
!python -m cerber.experiments.export --config configs/experiments/visdrone-n.yaml --format onnx

## Seraphim subset (класс drone)

Val режется из train. Официальный test не скачивается.

In [ ]:
!python -m cerber.experiments.prepare_seraphim --batches 1 --max-images 4000 --val-fraction 0.1
!python -m cerber.experiments.train --config configs/experiments/seraphim-subset.yaml

In [ ]:
!python -m cerber.experiments.val --config configs/experiments/seraphim-subset.yaml
!python -m cerber.experiments.export --config configs/experiments/seraphim-subset.yaml --format onnx

Скачай на ПК:
- `outputs/visdrone-n/weights/best.pt` → `configs/runtime-visdrone.yaml`
- `outputs/seraphim-n/weights/best.pt` → `configs/runtime-drone.yaml`
- локальный COCO-seg: `python -m cerber --config configs/runtime.yaml`